# 03. HF 원격 추론과 GitHub Stars 를 재다 — 첨부 자료는 카드와 달랐다

> 2026-09-03 · 이동원 · 결론 문서: [데이터파트 v3.2 고도화 설계](../../docs/데이터파트/version3.2/고도화_설계.md)

팀장이 조사한 고도화 자료에는 HuggingFace 모델 후보가 표 4개에 걸쳐 있었습니다.
그 표는 *"라이선스가 MIT 라 상업 이용이 자유롭다"* 처럼 적혀 있었는데, 우리 제약은 셋입니다.

1. **모델을 로컬에 내려받지 않는다** (transformers + torch 가 2GB 를 넘는다) → 원격 추론
2. **생성형은 쓰지 않는다** — 인코더까지
3. **라이선스가 모델 카드에 명시된 것만** 쓴다 (조사/도구선정 §4)

문서를 옮겨 적어서는 셋 중 하나도 확인되지 않습니다. 그래서 **서버에 물었습니다** —
`scripts/_probe_hf_inference.py` 가 카드 API 와 원격 추론 엔드포인트를 실제로 두드려
`reports/hf_inference_probe.json` 을 남겼고, 이 노트북은 그 결과를 읽습니다.

이 폴더의 규칙대로 **재현이 먼저**입니다 — 첨부가 말한 것을 그대로 재 보고, 다른 곳을 잽니다.

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

# 1~4절은 **권한을 켜기 전(11:27)** 실측을 읽는다. 켠 뒤 결과는 7절에서 따로 읽는다.
probe = json.loads((ROOT / "reports" / "hf_inference_probe_2026-09-03_before-permission.json")
                   .read_text(encoding="utf-8"))
print("실측 시각:", probe["measured_at_kst"], "· 후보", len(probe["rows"]), "종")
print("원격 추론 주소:", probe["router"])

rows = []
for r in probe["rows"]:
    card = r["card"]
    inf = r["inference"]
    router = inf.get("router", {}).get("status") if isinstance(inf.get("router"), dict) else None
    live = [p for p, v in (card.get("providers") or {}).items() if v.get("status") == "live"]
    rows.append({
        "모델": r["model"],
        "첨부 라이선스": r["claimed_license"],
        "카드 라이선스": card.get("license_tag") or card.get("license_card") or "미표기",
        "파이프라인": card.get("pipeline_tag"),
        "인코더": r["encoder"],
        "제공자(live)": ",".join(live) or "-",
        "호출": router if router is not None else "-",
        "판정": r["verdict"],
    })
df = pd.DataFrame(rows)
df

실측 시각: 2026-09-03T11:27:25+09:00 · 후보 21 종
원격 추론 주소: https://router.huggingface.co/hf-inference/models/{id}


,모델,첨부 라이선스,카드 라이선스,파이프라인,인코더,제공자(live),호출,판정
0,ProsusAI/finbert,MIT,미표기,text-classification,True,hf-inference,403,❌ 라이선스 미표기 (제약 3)
1,yiyanghkust/finbert-tone,MIT,미표기,text-classification,True,-,403,❌ 라이선스 미표기 (제약 3)
2,BAAI/bge-m3,MIT,mit,sentence-similarity,True,-,403,❌ mit · 원격 제공 안 함 (403 · 제공자 없음)
3,HuggingFaceFW/fineweb-edu-classifier,Apache-2.0,apache-2.0,text-classification,True,hf-inference,403,🟡 apache-2.0 · 제공자 ['hf-inference'] 인데 토큰 권한 없음(403)
4,jhgan/ko-sroberta-multitask,Apache-2.0,미표기,sentence-similarity,True,hf-inference,403,❌ 라이선스 미표기 (제약 3)
5,snunlp/KR-FinBert-SC,CC-BY-NC-SA-4.0,미표기,text-classification,True,hf-inference,403,❌ 라이선스 미표기 (제약 3)
6,gliner-community/gliner_medium-v2.5,Apache-2.0,apache-2.0,token-classification,True,-,403,❌ apache-2.0 · 원격 제공 안 함 (403 · 제공자 없음)
7,knowledgator/gliner-relex-large,Apache-2.0,미표기,NaN,True,-,403,❌ 카드 없음 (404)
8,microsoft/table-transformer-structure-recognition,MIT,mit,object-detection,True,-,-,🟡 mit · 호출 안 함 · 제공자 없음
9,nlpai-lab/KURE-v1,MIT,mit,feature-extraction,True,hf-inference,403,🟡 mit · 제공자 ['hf-inference'] 인데 토큰 권한 없음(403)


## 1. 발견 ① — 첨부의 라이선스 표기가 모델 카드와 다르다

첨부는 18종에 라이선스를 적어 두었습니다. 카드의 `license:` 태그(또는 `cardData.license`)와
맞대어 봅니다. **"미확인"** 으로 넣은 3종은 저장소 클라이언트(`ingest/clients/hf_data.py`)가
쓰던 모델과 다국어 대안이라 첨부에 없던 것입니다.

In [2]:
def _norm(s):
    return str(s or "").lower().replace("-", "").replace("_", "").replace(" ", "")

claimed = df[df["첨부 라이선스"] != "(미확인)"].copy()
claimed["일치"] = [
    _norm(a) == _norm(b) and b != "미표기"
    for a, b in zip(claimed["첨부 라이선스"], claimed["카드 라이선스"], strict=True)
]
mismatch = claimed[~claimed["일치"]][["모델", "첨부 라이선스", "카드 라이선스"]]
print(f"첨부에 라이선스가 적힌 {len(claimed)}종 중 카드와 다른 것 {len(mismatch)}종")
mismatch

첨부에 라이선스가 적힌 18종 중 카드와 다른 것 8종


,모델,첨부 라이선스,카드 라이선스
0,ProsusAI/finbert,MIT,미표기
1,yiyanghkust/finbert-tone,MIT,미표기
4,jhgan/ko-sroberta-multitask,Apache-2.0,미표기
5,snunlp/KR-FinBert-SC,CC-BY-NC-SA-4.0,미표기
7,knowledgator/gliner-relex-large,Apache-2.0,미표기
16,rbehzadan/ReaderLM-v2,MIT,미표기
18,facebook/nougat-base,MIT,cc-by-nc-4.0
19,openai/whisper-large-v3,MIT,apache-2.0


8종이 다릅니다. 다섯은 카드에 **라이선스 태그 자체가 없고**, `nougat-base` 는 MIT 가 아니라
**cc-by-nc-4.0**, `whisper-large-v3` 는 apache-2.0, `gliner-relex-large` 는 **카드 자체가 없습니다**(404).

첨부가 1순위로 꼽은 한국어 금융 감성 모델 `snunlp/KR-FinBert-SC` 는 첨부가 *"CC-BY-NC-SA 4.0"*
이라 적었지만 카드엔 아무 라이선스도 없습니다 — [조사/모델벤치마킹 §5](../../docs/조사/모델벤치마킹.md)
가 2026-09-02 에 같은 이유로 배제한 모델입니다. **검색 결과·블로그·GitHub 저장소의 LICENSE 는
가중치의 라이선스가 아닙니다.** (`ProsusAI/finBERT` 저장소는 Apache-2.0 인데 HF 카드는 미표기 —
아래 Stars 표에서 다시 봅니다.)

## 2. 발견 ② — 호출한 14종이 전부 403 이었다. 모델이 아니라 **토큰** 문제다

인코더 16종 중 시험 문장이 있는 14종에 실제로 보냈더니(비전 모델·베이스 인코더 2종은
보낼 문장이 없어 뺐습니다) 전부 `403` 이었습니다. 본문은 하나였습니다.

In [3]:
bodies = {}
for r in probe["rows"]:
    router = r["inference"].get("router") if isinstance(r["inference"], dict) else None
    if isinstance(router, dict) and router.get("status") == 403:
        bodies.setdefault(router.get("body", "")[:200], []).append(r["model"])
for body, models in bodies.items():
    print(f"{len(models)}종 →", body)

14종 → {"error": "This authentication method does not have sufficient permissions to call Inference Providers on behalf of user data-student"}


*"This authentication method does not have sufficient permissions to call Inference Providers"* —
토큰 권한입니다. `whoami-v2` 로 우리 토큰이 무엇을 할 수 있는지 봅니다. **토큰 값은 출력하지 않습니다.**

⚠️ 아래 셀은 **지금 시점**의 권한을 보여 줍니다. 11:27 실측 당시에는 `repo.*`·`discussion.write`
뿐이었고, 팀장이 11:5x 에 권한을 켰으므로 다시 실행하면 `inference` 항목이 보입니다.

In [4]:
def _token():
    for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():
        if line.startswith("HUGGINGFACE_ACCESS_TOKEN="):
            return line.partition("=")[2].strip().strip('"').strip("'")
    raise SystemExit("HUGGINGFACE_ACCESS_TOKEN 이 .env 에 없다")

HEADERS = {"Authorization": "Bearer " + _token()}
who = requests.get("https://huggingface.co/api/whoami-v2", headers=HEADERS, timeout=30).json()
tok = who.get("auth", {}).get("accessToken", {})
print("계정:", who.get("name"), "· 조직:", [o.get("name") for o in who.get("orgs", [])])
print("토큰 종류:", tok.get("role"), "· 이름:", tok.get("displayName"))
print("전역 권한:", tok.get("fineGrained", {}).get("global"))
for scope in tok.get("fineGrained", {}).get("scoped", []):
    print(f"  {scope['entity']['type']:5s} {scope['entity']['name']:16s} → {scope['permissions']}")

계정: data-student · 조직: ['qurious-quant']
토큰 종류: fineGrained · 이름: qurious-quant-token
전역 권한: ['discussion.write', 'post.write']
  org   qurious-quant    → ['repo.content.read', 'repo.access.read', 'discussion.write', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'org.read', 'org.write', 'org.billing.read', 'collection.read', 'collection.write', 'resourceGroup.write', 'job.write', 'org.serviceAccounts.read']
  user  data-student     → ['repo.content.read', 'repo.access.read', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'user.webhooks.read', 'user.webhooks.write', 'collection.read', 'collection.write', 'discussion.write', 'user.billing.read', 'job.write', 'user.notifications.read', 'user.notifications.write']


11:27 에는 `repo.*` 와 `discussion.write` 뿐이고 **`inference.*` 가 없었습니다.** Fine-grained
토큰은 "Make calls to Inference Providers" 를 따로 켜야 합니다. 서버는 6종을 `live` 로
답했으므로(아래 4절) 권한만 켜면 돌 것으로 예상했고, **7절에서 실제로 확인**했습니다.

## 3. 발견 ③ — "제공한다" 는 표시가 몇 분 사이에 흔들린다

`BAAI/bge-m3` 는 11:2x 첫 조회에서 `hf-inference: live` 였는데, 스크립트를 다시 돌린 11:27 에는
`error` 였습니다. 지금 다시 봅니다.

In [5]:
for model in ("BAAI/bge-m3", "yiyanghkust/finbert-tone", "nlpai-lab/KURE-v1"):
    r = requests.get(f"https://huggingface.co/api/models/{model}",
                     params={"expand[]": ["inference", "inferenceProviderMapping"]},
                     headers=HEADERS, timeout=30).json()
    mapping = r.get("inferenceProviderMapping") or {}
    if isinstance(mapping, dict):
        mapping = {k: v.get("status") for k, v in mapping.items()}
    print(f"{model:28s} inference={r.get('inference')!s:6s} providers={mapping}")

BAAI/bge-m3                  inference=None   providers={'hf-inference': 'error'}


yiyanghkust/finbert-tone     inference=None   providers={'hf-inference': 'error'}


nlpai-lab/KURE-v1            inference=warm   providers={'hf-inference': 'live'}


그래서 설계는 **한 번 조회한 값에 걸지 않습니다.** 기존 클라이언트가 실패를 예외가 아니라
`{"available": False}` 로 돌려주는 규약을 그대로 쓰고, 신호가 없는 날의 칸은 0 이 아니라 NaN 입니다.

## 4. 세 제약을 통과하는 것은 몇 종인가

In [6]:
def bucket(row):
    if not row["인코더"]:
        return "❌ 생성형 (제약 2)"
    if row["카드 라이선스"] == "미표기":
        return "❌ 라이선스 미표기 (제약 3)"
    if row["제공자(live)"] == "-":
        return "❌ 원격 제공 없음 (제약 1)"
    return "✅ 통과 (권한만 있으면 원격으로 돈다)"

df["구분"] = df.apply(bucket, axis=1)
print(df["구분"].value_counts().to_string())
print()
df[df["구분"].str.startswith("✅")][["모델", "카드 라이선스", "파이프라인", "제공자(live)"]]

구분
✅ 통과 (권한만 있으면 원격으로 돈다)    6
❌ 라이선스 미표기 (제약 3)         5
❌ 원격 제공 없음 (제약 1)         5
❌ 생성형 (제약 2)              5



,모델,카드 라이선스,파이프라인,제공자(live)
3,HuggingFaceFW/fineweb-edu-classifier,apache-2.0,text-classification,hf-inference
9,nlpai-lab/KURE-v1,mit,feature-extraction,hf-inference
10,kakaobank/kf-deberta-base,mit,fill-mask,hf-inference
13,joeddav/xlm-roberta-large-xnli,mit,zero-shot-classification,hf-inference
14,intfloat/multilingual-e5-large,mit,feature-extraction,"hf-inference,deepinfra"
15,sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2,apache-2.0,sentence-similarity,hf-inference


6종입니다. 이 가운데 **한국어 금융 감성 라벨을 바로 주는 모델은 없습니다.** 그래서 설계는
제로샷(`xlm-roberta-large-xnli` · MIT · 저장소 클라이언트 기본값) 으로 시작하고, 안 되면
임베딩(`KURE-v1` · MIT) + 로컬 로지스틱 회귀로 갑니다 — 파인튜닝이 아니라 다운로드 제약을 지킵니다.
`kf-deberta-base` 는 fill-mask 라 그대로는 라벨을 못 냅니다.

## 5. GitHub Stars — 벤치마킹 18곳 (GraphQL 단발 1회)

강사님 저장소를 포함해 우리 설계가 참고하거나 채택할 후보 18곳을 **한 번의 GraphQL 쿼리**로 쟀습니다
(`rateLimit cost 1`). 이 노트북은 그 결과 파일을 읽을 뿐 **API 를 다시 부르지 않습니다** —
반복 호출 금지 규약(AGENTS.md 1.5)입니다.

In [7]:
stars = json.loads((ROOT / "reports" / "github_stars_probe.json").read_text(encoding="utf-8"))
print("실측 시각:", stars["measured_at_kst"], "·", stars["method"])
sdf = pd.DataFrame(stars["rows"]).sort_values("stars", ascending=False)
sdf["license"] = sdf["license"].fillna("미표기")
sdf["pushed_at"] = sdf["pushed_at"].str[:10]
sdf[["repo", "stars", "license", "pushed_at", "language"]].reset_index(drop=True)

실측 시각: 2026-09-03T11:26:27+09:00 · GitHub GraphQL 1 call (gh api graphql) · aliases r00..r17 · rateLimit cost 1


,repo,stars,license,pushed_at,language
0,fastapi/fastapi,102040,MIT,2026-09-01,Python
1,nektos/act,71765,MIT,2026-08-09,Go
2,microsoft/qlib,48214,MIT,2026-09-02,Python
3,streamlit/streamlit,45671,Apache-2.0,2026-09-03,Python
4,PrefectHQ/prefect,23765,Apache-2.0,2026-09-03,Python
5,AI4Finance-Foundation/FinRL,16200,MIT,2026-07-13,Jupyter Notebook
6,Farama-Foundation/Gymnasium,12451,MIT,2026-08-29,Python
7,fivetran/great_expectations,11766,Apache-2.0,2026-09-02,Python
8,huggingface/text-embeddings-inference,5036,Apache-2.0,2026-07-24,Rust
9,unionai-oss/pandera,4447,MIT,2026-09-02,Python


**채택 셋** — `tox-dev/filelock`(MIT · 갱신 파이프라인 잠금) · `nektos/act`(MIT · Actions 를
push 없이 로컬에서 검증) · `Farama-Foundation/Gymnasium`(MIT · 관측 규격 계약). 나머지는
참고이거나 우리에게 이미 있는 것(FastAPI · huggingface_hub · FinanceDataReader)입니다.
`ProsusAI/finBERT` 저장소는 Apache-2.0 인데 HF 모델 카드는 미표기 — 1절의 어긋남이 이렇게 생깁니다.
강사님 저장소 `edumgt/quant-test-20260828` 은 Stars 0 · 라이선스 미표기이지만 강사님이 직접
가져다 쓰라고 허락하셨습니다 ([도입 설계](../../docs/데이터파트/version3.2/강사님저장소_분석과_도입설계.md)).

## 6. 권한을 켠 뒤 — 재실측 (11:53)

팀장이 토큰에 "Make calls to Inference Providers" 를 켰습니다. 같은 스크립트를 다시 돌린
결과(`reports/hf_inference_probe.json`)를 읽습니다. 이번엔 **무엇이 200 으로 돌았고, 얼마나
걸렸고, 무엇이 여전히 안 되는지**가 질문입니다.

In [8]:
after = json.loads((ROOT / "reports" / "hf_inference_probe.json").read_text(encoding="utf-8"))
print("재실측 시각:", after["measured_at_kst"])
rows = []
for r in after["rows"]:
    inf = r["inference"].get("router") if isinstance(r["inference"], dict) else None
    if not isinstance(inf, dict):
        continue
    card = r["card"]
    rows.append({
        "모델": r["model"],
        "카드 라이선스": card.get("license_tag") or card.get("license_card") or "미표기",
        "호출": inf.get("status"),
        "ms": inf.get("ms"),
        "응답 앞부분": (inf.get("body") or "")[:70],
    })
adf = pd.DataFrame(rows).sort_values(["호출", "ms"])
adf

재실측 시각: 2026-09-03T11:54:47+09:00


,모델,카드 라이선스,호출,ms,응답 앞부분
0,ProsusAI/finbert,미표기,200,322,"[[{""label"": ""positive"", ""score"": 0.9518615007400513}, {""..."
12,intfloat/multilingual-e5-large,mit,200,458,"""vector[1024]"""
11,joeddav/xlm-roberta-large-xnli,mit,200,1035,"{""sequence"": ""삼성전자 2분기 영업이익이 시장 예상을 웃돌았다."", ""labels"": [""..."
5,snunlp/KR-FinBert-SC,미표기,200,3682,"[[{""label"": ""positive"", ""score"": 0.9992161989212036}, {""..."
9,kakaobank/kf-deberta-base,mit,200,6704,"[{""score"": 0.2577744722366333, ""token"": 311, ""token_str""..."
3,HuggingFaceFW/fineweb-edu-classifier,apache-2.0,200,11986,"[[{""label"": ""LABEL_0"", ""score"": -0.030611343681812286}]]"
8,nlpai-lab/KURE-v1,mit,200,24864,"""vector[1024]"""
7,knowledgator/gliner-relex-large,미표기,400,240,"{""error"": ""Model not supported by provider hf-inference""}"
6,gliner-community/gliner_medium-v2.5,apache-2.0,400,253,"{""error"": ""Model not supported by provider hf-inference""}"
10,FISA-conclave/klue-roberta-news-sentiment,apache-2.0,400,345,"{""error"": ""Model not supported by provider hf-inference""}"


**200 이 7종, 400 이 7종.** 200 가운데 라이선스가 명시된 것은 5종 —
`fineweb-edu-classifier` · `KURE-v1` · `kf-deberta-base` · `xlm-roberta-large-xnli` · `multilingual-e5-large`.
`finbert`·`KR-FinBert-SC` 도 돌지만 카드에 라이선스가 없어 못 씁니다.

지연은 **콜드 스타트가 좌우**합니다 — 이미 떠 있던 `e5` 0.46초 · `xlm-roberta` 1.0초, 처음 뜨는
`KURE-v1` 24.9초 · `fineweb` 12.0초. `wait_for_model` 을 켜 두면 기다려 주지만, 갱신 파이프라인
(고도화 설계 §3)의 타임아웃은 이 값으로 잡아야 합니다.

400 은 두 종류입니다. `Model not supported by provider hf-inference`(gliner 둘 · klue-roberta-news-sentiment)
는 **정말 제공 안 함**이고, `SentenceSimilarityPipeline … missing 'sentences'`(bge-m3 · ko-sroberta · MiniLM)
는 **경로가 다른 것**입니다 — 저장소 클라이언트 `hf_data.py` 가 2026-08-04 에 같은 것을 실측해
`/pipeline/feature-extraction` 경로를 씁니다. 그 경로로 다시 두드린 결과:

In [9]:
fe = json.loads((ROOT / "reports" / "hf_inference_probe_feature_extraction_path.json")
                .read_text(encoding="utf-8"))
print("경로:", fe["path"])
for m, v in fe["rows"].items():
    print(f"{m:60s} {v['status']!s:5s} {v['ms']:7,d}ms  {str(v['body'])[:60]}")

경로: /pipeline/feature-extraction
BAAI/bge-m3                                                  504   120,320ms  <html>
<head><title>504 Gateway Time-out</title></head>
<b
jhgan/ko-sroberta-multitask                                  200     5,822ms  vector[1x768]
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  200       302ms  vector[1x384]


한국어 제로샷은 실제로 한국어를 읽습니다 — `xlm-roberta-large-xnli` 에 "삼성전자 2분기 영업이익이
시장 예상을 웃돌았다." 를 주고 후보 라벨 `긍정·부정·중립` 을 줬더니 **긍정 0.988**. 설계 §2.2 의
길 A 가 종이 위에서 내려왔습니다.

## 7. 배운 것

1. **첨부의 라이선스는 카드가 아니다.** 18종 중 8종이 달랐다(표기 불일치 7 + 카드 없음 1). 카드만 믿는다.
2. **403 은 셋 중 하나다** — 모델이 없다 · 서버가 제공하지 않는다 · 내 토큰 권한이 없다. 본문을 읽어야 갈라진다. 이번엔 셋째였다.
3. **400 도 둘로 갈라진다** — "제공 안 함" 과 "경로가 다르다". 후자는 400 을 보고 버리면 멀쩡한 모델을 잃는다.
4. **콜드 스타트는 25초까지.** 타임아웃과 `wait_for_model` 은 이 값으로 정한다.
5. **원료가 0행이면 모델은 없다.** `dart_disclosure` 는 표만 있다. 모델 표보다 원료 표가 먼저다.
6. **API 는 단발로 잰다.** GraphQL 1회 · 카드 API 21회 · 추론 14+3회. 반복 호출은 규약 위반이다.

다음: 단가(크레딧 소모)를 재고, v11 `text_signal` 에서 공시 제목 배치 64건이 한 콜에 드는지 본다.